<a href="https://colab.research.google.com/github/dEVT20034/f1-podium-predictor/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastf1

In [ ]:
import fastf1
import pandas as pd
import requests,time
import os

In [ ]:
url="https://api.jolpi.ca/ergast/f1/2024/results"
r=requests.get(url,params={"limit":100})
data=r.json()["MRData"]

print("Total rows for 2024:", data["total"])
race = data["RaceTable"]["Races"][0]
print(race["raceName"])
print(race["Results"][0])

In [ ]:
#Now we are fetching results from all seasons
def fetch_all(endpoints):
  races,offset=[],0
  while True:
    r=requests.get(f"https://api.jolpi.ca/ergast/f1/{endpoints}.json",
                   params={"limit":100,"offset":offset})
    r.raise_for_status()
    d=r.json()["MRData"]
    races+=d["RaceTable"]["Races"]
    offset+=100
    if offset>=int(d["total"]):
      return races
    time.sleep(0.5)

rows=[]
for season in range(2018,2026):
  for race in fetch_all(f"{season}/results"):
    for res in race["Results"]:
      rows.append({
          "season": int(race["season"]),
          "round":int(race['round']),
          "date":race['date'],
          "circuitId":race['Circuit']['circuitId'],
          "driverId":res["Driver"]["driverId"],
          "constructorId":res["Constructor"]["constructorId"],
          "grid":int(res["grid"]),
          "finish_pos":pd.to_numeric(res["position"],errors="coerce"),
          "points":float(res["points"]),
          "status":res["status"],
      })

results=pd.DataFrame(rows)
results.to_csv("results.csv",index=False)
print(results.shape)
results.head()

In [ ]:
#Now lets fetch data for Qualifying
rows=[]
for season in range(2018,2026):
  for race in fetch_all(f"{season}/qualifying"):
    for q in race["QualifyingResults"]:
      rows.append({
          "season":int(race['season']),
          "round":int(race['round']),
          "driverId": q["Driver"]["driverId"],
          "quali_pos": pd.to_numeric(q["position"], errors="coerce"),
      })

quali = pd.DataFrame(rows)
quali.to_csv("quali.csv", index=False)
print(quali.shape)
quali.head()

In [ ]:
#Here we will fetch our weather data
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

race_keys = results[["season", "round"]].drop_duplicates().values.tolist()

rows = []
for i, (season, rnd) in enumerate(race_keys):
    try:
        ses = fastf1.get_session(int(season), int(rnd), "R")
        ses.load(laps=False, telemetry=False, weather=True, messages=False)
        w = ses.weather_data
        rows.append({
            "season": int(season), "round": int(rnd),
            "air_temp": w["AirTemp"].mean(),
            "track_temp": w["TrackTemp"].mean(),
            "humidity": w["Humidity"].mean(),
            "wind_speed": w["WindSpeed"].mean(),
            "rain": int(w["Rainfall"].any()),
        })
    except Exception as e:
        print(f"Failed {season} R{rnd}: {e}")
    if i % 20 == 0:
        print(f"{i}/{len(race_keys)} done")

weather = pd.DataFrame(rows)
weather.to_csv("weather.csv", index=False)
print(weather.shape)
weather.head()

In [ ]:
#We will combine our all datasets(like:race_results, qualifying_results, and weather_dataset)
df=(results.merge(quali,on=["season","round","driverId"],how="left")
  .merge(weather, on=["season", "round"], how="left"))

df["date"]=pd.to_datetime(df["date"])
df=df.sort_values(["date","finish_pos"]).reset_index(drop=True)

print(df.shape)
print(df.isna().sum())
df.head()

In [ ]:
print(weather.shape)
print(weather.groupby("season").size())